# Decoded quantum interferometry for Max-XORSAT

All 15 source stages, the 15-bit unary example, both six-bit Dicke states, the six-cycle Max-XORSAT graph and all 22 lookup entries. Use 2,048 shots for each preparation and 10,000 DQI samples.

Prepare the unary example from its known classical count and use a reversible preparation specialized to the specified unary basis input for the second Dicke example. Regenerate the incidence and decoder tables and solve the original three-by-three eigenproblem. The native circuit keeps both output words and tests the zero-error word directly from measured results.

## Compose the coherent encoder and decoder

The editable DQI root below requests symmetric amplitudes, sign phases, GF(2)
encoding, the lookup decoder and final Hadamards. The classical eigenproblem
and source graph generate its binding tables. FlagQuantum constructs the
controlled decoding operations and manages their work bits; the user still
provides the code, supported error weights and decoder semantics.

The unary-to-Dicke helper is specialized to a known basis input, not a general
coherent weight-conversion primitive. The 22-entry decoder covers weight-zero
through weight-two errors; it is not a scalable decoder inferred for any graph.
Try another normalized combination of those supported weights through
`symmetric_amplitudes`, keeping the table valid. Check the entire y=0 marginal,
not merely a finite sample with no observed error. Compiler-work cleanup and
the logical decoded error register are different checks.


## Run the native unary and Dicke preparations

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.overview.model import (
    DQI_SHOTS,
    EDGES,
    NODES,
    majority_errors,
)
from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.overview.plots import (
    save_graph,
    save_scores,
)
from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.overview.workflow import (
    run_workflow,
)

folder = Path("artifacts/development/dqi-workflow")
result = run_workflow()
print("Unary observation:", result.unary)


## Read the six-bit weight-four Dicke sample

In [ ]:
assert all(word.bit_count() == 4 for word, _ in result.dicke_four)


## Read the unary-to-Dicke sample

In [ ]:
assert all(word.bit_count() == 2 for word, _ in result.dicke_two)
print("Dicke support sizes:", len(result.dicke_four), len(result.dicke_two))


## Inspect the six-column parity encoder

In [ ]:
print("GF(2) columns:", result.model.columns)


## Inspect the coherent decoder sequence

In [ ]:
print(
    "Prepare symmetric weights, phase the clauses, encode the syndrome and uncompute the error word."
)


## Generate the source graph

In [ ]:
print("Source graph:", NODES, EDGES)
display(Image(filename=str(save_graph(folder / "graph.png"))))


## Enumerate all assignment scores

In [ ]:
print("Score support:", sorted(set(result.model.objective)))


## Enumerate the weight-zero through weight-two errors

In [ ]:
assert len(result.model.syndrome_table) == 22


## Inspect the actual decoder entries

In [ ]:
print("Syndrome lookup:", result.model.syndrome_table)


## Compare the illustrative majority rule

In [ ]:
print(
    "Decoder entries and majority misses:",
    len(result.model.syndrome_table),
    majority_errors(result.model),
)


## Regenerate the principal weight eigenvector

In [ ]:
print("Optimal weight amplitudes:", result.model.weights)


## Inspect the original-size native program

In [ ]:
import math
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QArray,
    QBit,
    allocate,
    inplace_prepare_amplitudes,
    phase,
    qfunc,
)
from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.main import (
    decode,
    gf2_product,
    symmetric_amplitudes,
)
from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.overview.model import (
    SIGNS,
)


@qfunc
def decoded_interference(
    amplitudes: list[float],
    matrix: list[int],
    signs: list[int],
    syndrome_bits: list[int],
    mask_bits: list[int],
    y: Output[QArray[QBit]],
    solution: Output[QArray[QBit]],
) -> None:
    allocate(6, y)
    allocate(6, solution)
    inplace_prepare_amplitudes(amplitudes, 0, y)
    for bit in range(6):
        phase(y[bit], math.pi * signs[bit])
    gf2_product(matrix, y, solution)
    decode(syndrome_bits, mask_bits, y, solution)
    for bit in range(6):
        H(solution[bit])


editable_program = synth.synthesize(
    decoded_interference,
    classical_inputs={
        "amplitudes": symmetric_amplitudes(result.model.weights),
        "matrix": result.model.matrix,
        "signs": SIGNS,
        "syndrome_bits": [
            word >> bit & 1
            for word, _ in result.model.syndrome_table
            for bit in range(6)
        ],
        "mask_bits": [
            mask >> bit & 1
            for _, mask in result.model.syndrome_table
            for bit in range(6)
        ],
    },
    max_qubits=15,
    max_workspace=3,
    memory_limit_bytes=64 * 1024**2,
)
print("Editable DQI resources:", editable_program.report["resources"])
print(
    "Decoded error marginal:",
    dict(editable_program.probabilities(fq.run(editable_program), registers=("y",))),
)
print("Native resources:", result.program.report["resources"])


## Read the actual finite sample

In [ ]:
print("DQI shots:", sum(count for _, count in result.dqi))
assert sum(count for _, count in result.dqi) == DQI_SHOTS


## Verify that the measured decoder cleared the error

In [ ]:
assert all(error == 0 for (error, _), _ in result.dqi)


## Compare score distributions and expectations

In [ ]:
print("Sampled DQI score:", result.mean_score)
print("Exact uniform score:", sum(result.model.objective) / len(result.model.objective))
display(Image(filename=str(save_scores(result, folder / "scores.png"))))
